# Default notebook

This default notebook is executed using a Lakeflow job as defined in resources/sample_job.job.yml.

In [0]:
# Set default catalog and schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
print(f"Schema {catalog}.{schema} is ready")

In [0]:
from datetime import timedelta
from pyspark.sql import functions as F
from pyspark.sql import Row

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
target = f"{catalog}.{schema}.nyc_trips"

# Find the latest pickup timestamp currently in the table
max_pickup = (
    spark.read.table(target)
    .agg(F.max("tpep_pickup_datetime"))
    .collect()[0][0]
)

# Start incremental rows just past whatever exists (or a sensible base if empty)
base_ts = max_pickup if max_pickup is not None else F.to_timestamp(F.lit("2016-01-01 00:00:00")).alias("x")
if max_pickup is None:
    base_ts = spark.sql("SELECT to_timestamp('2016-01-01 00:00:00') AS t").collect()[0][0]

# Build N new rows with incrementing timestamps, distances, and fares
n_new_rows = 10
new_rows = [
    Row(
        tpep_pickup_datetime=base_ts + timedelta(hours=i + 1),
        tpep_dropoff_datetime=base_ts + timedelta(hours=i + 1, minutes=35),
        trip_distance=float(5 + i),
        fare_amount=float(20 + 2 * i),
        pickup_zip=10282,
        dropoff_zip=10171,
    )
    for i in range(n_new_rows)
]

df_new = spark.createDataFrame(new_rows)
df_new.write.mode("append").saveAsTable(target)

print(f"Inserted {n_new_rows} incremental rows (base pickup ts: {base_ts})")
print("New row count:", spark.read.table(target).count())


In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
target = f"{catalog}.{schema}.nyc_trips"

count = spark.sql(f"SELECT count(*) FROM {target}").collect()[0][0]
print(f"Row count in {target}: {count}")